# Custom SMC steps in `sequince`

The `sequince` framework can be used to construct custom sampling routines that execute an arbitrary cycle of steps until a termination criterion is reached. In this example, we'll explore some ways the user can incorporate their own logic into an SMC sampler.

## Step structure

`sequince` implements SMC steps via `sequince.steps.base.SMCStep`. Steps with a specific function are derived by subclassing this and overriding the `apply` method, which takes the following inputs:

* **state**: An instance of `sequince.state.ParticleState`, which describes the current state of the sampler.
* **context**: An instance of `sequince.steps.base.StepContext` which allows for additional information to be shared between (and updated by) different sampling steps.
* **rng**: An RNG instance to provide a common interface for random number generation.

The `StepContext` enables information to be shared via the `shared` dictionary attribute. The `metadata` dictionary attribute should be used for sharing information that can be mutated. The `outputs` dictionary can be used to store outputs from steps.


## An example problem

As a simple example, consider a Gaussian pulse $h(t)$ in normally-distributed additive noise $n(t)$ of variance $\sigma^2$, such that $d(t) = h(t) + n(t)$. We wish to infer the amplitude $A$ and location $\mu$ of the pulse, which are defined by

$$ h(t; \mu ; A) = A e^{-(t- \mu)^2} $$

The log-likelihood function given the pulse amplitude $A$ and location $\mu$ is

$$ \log \mathcal{L}(d \mid \mu, A) = -\frac{(h(t ; \mu ; A) - d)^2}{2 \sigma^2} + C$$

where $C$ is an additive constant independent of $\mu$ and $A$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

A_true = 2
mu_true = 7

noise_variance = 0.5

t_data = np.linspace(0, 20, 501)
h_t = A_true * np.exp(-0.5 * (t_data - mu_true) ** 2)
noise = np.random.normal(0, np.sqrt(noise_variance), size=t_data.shape)

d_data = h_t + noise

plt.figure(figsize=(6, 3))
plt.plot(t_data, d_data, "o", label="Data", markersize=4)
plt.plot(t_data, h_t, "r-", label="Signal", linewidth=2)
plt.legend()
plt.show()

## Basic workflow

We can sample the posterior distribution corresponding to this with `sequince` via inverse temperature β annealing. A basic example of such a workflow is:

In [ ]:
from sequince import ParticleState, SequentialMonteCarlo, TemperedPosterior
from sequince.steps import (
    AdaptiveTemperedWeighting,
    EmceeMutation,
    SystematicResampling,
)


def log_likelihood(particles, context):
    # we use the context to share data and other necessary info with the log-likelihood function
    t = context.shared["t_data"]
    d = context.shared["d_data"]
    var = context.shared["noise_variance"]

    templates = particles[:, 0] * np.exp(
        -0.5 * (t[:, None] - particles[:, 1]) ** 2
    )
    return -0.5 * ((templates - d[:, None]) ** 2).sum(axis=0) / var


def log_prior(particles, context):
    # similarly, we can use the context to share prior bounds with the log-prior function
    bounds = context.shared["bounds"]
    return np.where(
        np.all((particles >= bounds[0]) & (particles <= bounds[1])),
        0.0,
        -np.inf,
    ).sum(axis=-1)


rng = np.random.default_rng(seed=123)

bounds = np.array(
    [
        [1.0, 5.0],
        [
            1.0,
            19.0,
        ],
    ]
).T

n_particles = 200
initial_particles = rng.uniform(
    low=bounds[0], high=bounds[1], size=(n_particles, bounds.shape[0])
)

initial_log_weights = np.full(n_particles, -np.log(n_particles))
initial_state = ParticleState(
    particles=initial_particles, log_weights=initial_log_weights
)

# Small helper to create the tempered posterior
# This allows the different steps to use either the log prior, log likelihood,
# or the full tempered posterior as needed.
posterior = TemperedPosterior(
    log_prior=log_prior,
    log_likelihood=log_likelihood,
)

smc = SequentialMonteCarlo(
    steps=[
        AdaptiveTemperedWeighting(posterior, target_ess_ratio=0.8),
        SystematicResampling(always_resample=True),
        EmceeMutation(
            posterior,
            n_steps=20,
            vectorize=True,
            run_kwargs=dict(skip_initial_state_check=True),
        ),
    ]
)

# Shared context for the SMC run
# If information is needed during the sampling steps (as defined above), dump it here.
shared_context = {
    "t_data": t_data,
    "d_data": d_data,
    "noise_variance": noise_variance,
    "bounds": bounds,
}

results = smc.run(initial_state, rng=rng, shared_context=shared_context)

posterior_state = results.final_state

# Analyze the posterior particles
particles = posterior_state.particles
mean_estimate = np.mean(particles, axis=0)
cov_estimate = np.cov(particles, rowvar=False)
print(f"Posterior mean estimate: {mean_estimate}")
print(f"Posterior covariance estimate:\n{cov_estimate}")

## Adding custom steps

The above is sufficient for sampling the posterior, but maybe we want to introduce some additional features into our workflow. For instance, maybe we want to continuously plot the particle positions to a common figure, or print the best particle location every few iterations. This can be easily achieved by adding custom steps to the workflow, as we will now demonstrate.

In [ ]:
from sequince.steps.base import SMCStep


class MonitorBestPointStep(SMCStep):
    def apply(self, state, context, rng):
        particles = state.particles
        best_weight = np.argmax(state.log_weights)
        print(
            f"Best particle:, {particles[best_weight]} with log-weight {state.log_weights[best_weight]}"
        )
        return state  # A state must be returned


class ScatterPlotCurrentParticlePositions(SMCStep):
    def __init__(self, every_n_steps=1, always_final_step=True):
        self.every_n_steps = every_n_steps
        self.always_final_step = always_final_step

    def apply(self, state, context, rng):
        if context.t % self.every_n_steps == 0 or (
            context.metadata["stop_requested"] and self.always_final_step
        ):
            # Fetch the figure from the mutable context metadata
            fig = context.metadata["figure"]
            ax = fig.gca()
            particles = state.particles
            ax.scatter(
                particles[:, 0], particles[:, 1], alpha=0.5, label=context.t
            )
            return state  # A state must be returned
        else:
            return state


fig, ax = plt.subplots(figsize=(6, 6))

smc = SequentialMonteCarlo(
    steps=[
        AdaptiveTemperedWeighting(posterior, target_ess_ratio=0.8),
        SystematicResampling(always_resample=True),
        EmceeMutation(
            posterior,
            n_steps=20,
            vectorize=True,
            run_kwargs=dict(skip_initial_state_check=True),
        ),
        MonitorBestPointStep(),
        ScatterPlotCurrentParticlePositions(every_n_steps=3),
    ]
)

shared_context = {
    "t_data": t_data,
    "d_data": d_data,
    "noise_variance": noise_variance,
    "bounds": bounds,
}
# Include the figure in the metadata since it will be mutated
metadata = {
    "figure": fig,
}

results = smc.run(
    initial_state, rng=rng, shared_context=shared_context, metadata=metadata
)

ax.set_xlabel("Amplitude (A)")
ax.set_ylabel("Mean (mu)")
ax.set_title("Particle Positions")
ax.legend()
plt.show()

SMC steps can also modify the context, which is useful for constructing logic between different steps. As a basic example of this, we could define one step that monitors the iteration count and another that waits for a signal from that step before printing information. This could be achieved in a single step in this case, but for more general cases it can be easier to separate the two.

We use the `metadata` attribute of `StepContext` rather than `shared` since the latter is intended to be inmuatable.

In [ ]:
from sequince.steps.base import SMCStep


class MonitorStepIteration(SMCStep):
    def apply(self, state, context, rng):
        if context.t == 6:
            context.metadata["do_printing"] = True
        return state


class PrintingStep(SMCStep):
    def apply(self, state, context, rng):
        if context.metadata.get("do_printing", False):
            particles = state.particles
            mean_estimate = np.mean(particles, axis=0)
            print(f"Iteration {context.t}: Mean estimate: {mean_estimate}")
        return state


smc = SequentialMonteCarlo(
    steps=[
        AdaptiveTemperedWeighting(posterior, target_ess_ratio=0.8),
        SystematicResampling(always_resample=True),
        EmceeMutation(
            posterior,
            n_steps=20,
            vectorize=True,
            run_kwargs=dict(skip_initial_state_check=True),
        ),
        MonitorStepIteration(),
        PrintingStep(),
    ]
)

shared_context = {
    "t_data": t_data,
    "d_data": d_data,
    "noise_variance": noise_variance,
    "bounds": bounds,
}
metadata = {
    "do_printing": False,
}

results = smc.run(
    initial_state, rng=rng, shared_context=shared_context, metadata=metadata
)